# <p style= "text-align: center">From Fourier to wavelets</p>
## <p style= "text-align: center">Transforms, implementations and applications</p>

This notebook refreshes Fourier analysis and while doing so, highlights certain insights with great implications for signal analysis. We look at the different mathematical transforms, and see what we can learn from the continuous and discrete transform and the series decomposition. We will discuss the algorithmic implementation of the transforms through the fast Fourier transform and give some example problems where you may want to use Fourier analysis. With our refreshed insight in Fourier analysis, we will tackle time-frequency analysis of signals by introducing you to the Gabor and wavelet transforms. We briefly discuss some of the mathematics, but mainly aim to provide an intuitive understanding of what these transforms are and how you could use them. We end the notebook with some example problems where you may want to consider the Gabor or wavelet transform.

## 0) Set up workspace

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.signal import ShortTimeFFT
from scipy.signal.windows import gaussian
import pywt

import spectrogram_tools as st
from importlib import reload

In [ ]:
reload(st) 

## 1) Fourier analysis

Fourier analysis deals with representing or approximating general functions as sums of trigonometric functions or complex exponentials. Depending on whether the function is continuous or discrete in its domain, whether it is periodic or aperiodic in its domain, and whether the domain is finite, the Fourier analysis takes any of the five following forms:

- Continuous time Fourier series (CTFS): continuous, periodic functions on $\mathbb{R}$
- Continuous time Fourier transform (CTFT): continuous, aperiodic functions on $\mathbb{R}$
- Discrete time Fourier series (DTFS): discrete, periodic functions on $\mathbb{Z}$
- Discrete time Fourier transform (DTFT): discrete, aperiodic functions on $\mathbb{Z}$
- Discrete Fourier transform (DFT): discrete, aperiodic functions on $D \subset \mathbb{Z}$

The DFT is the only transform that can be computed numerically and is therefore the most important for our goal of eventually computing spectrograms by using similar transforms. However, we will first revisit some of the other transforms as they provide fundamental insights in why we would want to eventually use a wavelet transform to compute spectrograms. The DFT can be used to approximate the other transforms.

References:
- S. L. Brunton and J. N. Kutz, Data-Driven Science and Engineering: Machine Learning, Dynamical Systems, and Control, 2nd ed. Cambridge: Cambridge University Press, 2022. 
- https://en.wikipedia.org/wiki/Fourier_analysis

### A) Continuous time Fourier series

We start by considering the CTFS as it provides a beautiful geometric interpretation of Fourier analysis. The space of continuous functions is a vector space, which can be associated with an inner product.

$$
\langle f, g \rangle =\int_{0}^{T}f(t){\overline{g(t)}} \, \mathrm{d} t
$$

Such an inner product then allows to define the projection of a function onto another function, completely analogously to how you project vectors in the familiar Euclidean space.

$$
\operatorname{proj}_g (f) = \frac{\langle f, g \rangle}{\langle g, g \rangle} g
$$

Trigonometric functions of increasing frequency, i.e. $sin(k \frac{2\pi}{T} t)$ and $cos(k \frac{2\pi}{T} t)$, are orthogonal, meaning that the inner product of any of these functions with one another is zero unless they are identical. The set of these trigonometric functions forms a base for the space of continuous periodic functions, and hence all continuous periodic functions can be expressed as a (generally infinite) weighted sum of these functions. More compactly, the base of functions can be made up of complex exponentials $e^{ik\frac{2\pi}{T}t}$ which can can be expressed as $cos(k \frac{2\pi}{T} t) + i \, sin(k \frac{2\pi}{T} t)$ by Euler's formula.

To find this weighted sum, the function is projected onto each of the elements of the base and the projections are summed. This weighted sum is the Fourier series decomposition of the function and provides us with insight about the frequency content of a function. Below are two formulations, one with the explicit trigonometric base and one with the exponential base. They lead to identical results, so in what follows we will only use the exponential notation for convenience.

Exponential form:

$$
f(t) = \sum_{k=-\infty}^{\infty} c_{k} e^{ik\frac{2\pi}{T}t}
$$

$$
c_{k} = \frac{1}{T} \int_{0}^{T} \, f(t) \, e^{-ik\frac{2\pi}{T}t} \, dt
$$

Trigonometric form:

$$
f(t) = a_{0} + \sum_{k=1}^{\infty} \left(a_{k}\cos\left(k\frac{2\pi}{T}t\right) + b_{k}\sin\left(k\frac{2\pi}{T}t\right)\right)
$$

$$
\begin{aligned}
&a_{0}={\frac {1}{T}}\int_{0}^{T} f(t) \, dt & \\
&a_{k}={\frac {2}{T}}\int_{0}^{T} f(t) \cos\left(k\frac{2\pi}{T}t\right) \, dt ,\ & \textrm{for} ~k \geq 1\\
&b_{k}={\frac {2}{T}}\int_{0}^{T} f(t) \sin\left(k\frac{2\pi}{T}t\right)\, dt ,\ & \text{for} ~k \geq 1\\
\end{aligned}
$$

Both forms are equivalent, by the relations:

$$
\begin{aligned}
a_{0} &= c_{0} & \\
a_{k} &= c_{k}+c_{-k} \qquad & \textrm{for} ~k>0\\
b_{k} &= (c_{k}-c_{-k})/i \qquad & \textrm{for} ~k>0\\
\end{aligned}
$$

References: 
- https://en.wikipedia.org/wiki/Hilbert_space
- https://en.wikipedia.org/wiki/Fourier_series

In [ ]:
########################################
# Example Fourier series decomposition #
########################################

# Define function to be decomposed
npts = 1001
T = 1
t_axis = np.linspace(0, T, num=1001)
f = np.exp(-(t_axis-0.5)**2/0.05) # some Gaussian as an example of a smooth function
# f = (t_axis > (T/4)) * (t_axis < (3*T/4)) # some step function as an example of a discontinuous function, you will notice the Gibbs phenomenon if you increase K

# Define inner product
def inner_product(f, g, T):
    n = f.size
    res = np.sum(f*np.conjugate(g)) * (T/(n-1))
    return res

# Calculate coefficients explicitly
K = 2
basis_functions = []
coefficients = []
f_series = np.zeros_like(f, dtype=complex)

for k in np.arange(-K, K+1):
    g_k = np.exp(1j * k * (2*np.pi/T) * t_axis)
    c_k = (1/T) * inner_product(f, g_k, T)
    basis_functions.append(g_k)
    coefficients.append(c_k)
    f_series = f_series + c_k * g_k

# Make plot
fig, axs = plt.subplots(nrows=2, ncols=1)
fig.suptitle(f"Fourier series decomposition with {2*K+1} components")

axs[0].plot(t_axis, f, label="Original")
axs[0].plot(t_axis, f_series.real, label="Fourier series", linestyle="--")
axs[0].set_ylim(-0.3, 1.05)
axs[0].legend()
axs[0].grid()

for i in np.arange(2*K+1):
    component = coefficients[i] * basis_functions[i]
    k = int(i-K)
    if k > 0:
        axs[1].plot(t_axis, component.real, label=str(k), linestyle="--")
    else:
        axs[1].plot(t_axis, component.real, label=str(k))
axs[1].set_ylim(-0.3, 1.05)
axs[1].legend()
axs[1].grid()

fig.tight_layout()
plt.show()

### B) Continuous time Fourier transform

The CTFT is defined for any integrable (integral of function over entire domain exists and is finite), complex-valued function on the real line (or $\mathbb{R}^n$). There are a few forms of the transform and its inverse, leading to different normalization factors to make sure the transform is unitary. If a transform is unitary, the inner product of any two vectors/functions is equal to the inner product of the transform of those two vectors/functions. Unitarity means that Plancherel's theorem holds (conservation of energy). The simplest form is given below, where we will interpret $\xi$ as frequency expressed in Hertz if $t$ represents time in seconds. This form requires no normalization.

$$
\begin{aligned}
F(\xi)\ &= \ \int_{-\infty }^{\infty } f(t)\,e^{-i2\pi \xi t}\,dt \\
f(t) &= \int_{-\infty }^{\infty } F(\xi )\,e^{i2\pi \xi t}\,d\xi 
\end{aligned}
$$

The forward CTFT is basically the generalization of the coefficient calculation for the CTFS to a continuous domain of frequencies instead of a discrete set of frequencies. You can compare the expression for $F(\xi)$ given here to the expression for $c_k$ in the previous subsection. The inverse CTFT recovers the original function from $F(\xi)$ much like the CTFS represents a function as a decomposition into different frequency components. You can look at the expression for $f(t)$ as a series in the previous subsection and see that if you replace the sum by an integral, $c_k$ by $F(\xi)$ and $k/T$ by $\xi$, you end up with the inverse CTFT. The CTFT thus still informs us about the frequency content of a function.

An interesting property of the CTFT with important implications for signal processing is the uncertainty principle. This means that a function cannot be arbitrarily 'concentrated' or 'localized' in both the time and frequency domain. Formally, the following holds for any function $f$, which is assumed to be normalized so that $\int_{-\infty}^{\infty} \left|f(t)\right|^2 \,dt = 1$:

$$
\left( \int_{-\infty}^{\infty} t^2 \left|f(t)\right|^2 \,dt \right) \left( \int_{-\infty}^{\infty} \xi^2 \left|F(\xi)\right|^2 \,d\xi \right) \geq \frac{1}{16\pi^2}
$$

So the product of the variance of a function in the time domain with the variance of that same function transformed to the frequency domain is greater than or equal to some bound. This implies for instance that if we consider a stationary vibration signal, meaning that the frequency components do not change over time, we can compute more accurate power spectra (narrower peaks) if we use longer samples (wider in time domain). Stationary vibration signals occur in conservative linear systems. If you consider nonlinear systems and vibrations (or non-conservative or non-stationary systems), frequency components change over time. In this case, if you are interested in the frequency content at any given time, you cannot consider an arbitrarily long sample, meaning you can never completely accurately reconstruct the power spectrum. This means that later on, when we compute spectrograms, we will have to be careful in how we select certain windowing functions or wavelets, so as to find the optimal trade-off between time and frequency resolution. Below, you may find a script that illustrates this trade-off.

References:
- https://en.wikipedia.org/wiki/Fourier_transform

In [ ]:
######################################
# Illustration uncertainty principle #
######################################

# Define functions to be transformed and compared
npts_t = 8001
T = [-4, 4] # time domain
dt = (T[1]-T[0])/(npts_t-1)
t_axis = np.linspace(T[0], T[1], num=npts_t)
f_narrow = np.cos(2*np.pi*3*t_axis) * np.exp(-np.pi*t_axis**2) # 3 Hz cosine, modulated by narrow Gaussian
f_wide = np.cos(2*np.pi*3*t_axis) * np.exp(-np.pi*t_axis**2/9) # 3 Hz cosine, modulated by wide Gaussian
# f_narrow = np.cos(2*np.pi*3*t_axis) * ((t_axis >= -0.75)*(t_axis < 0.75)) # 3 Hz cosine, modulated by narrow step function
# f_wide = np.cos(2*np.pi*3*t_axis) * ((t_axis >= -2.5)*(t_axis < 2.5)) # 3 Hz cosine, modulated by wide step function

# Calculate Fourier transforms
npts_xi = 12001
XI = [-6, 6] # frequency domain
xi_axis = np.linspace(XI[0], XI[1], num=npts_xi)
F_narrow = np.zeros(npts_xi, dtype=complex)
F_wide = np.zeros(npts_xi, dtype=complex)

for k in np.arange(npts_xi):
    g_k = np.exp(-1j * 2*np.pi * xi_axis[k] * t_axis)
    F_narrow[k] = np.sum(f_narrow * g_k) * dt
    F_wide[k] = np.sum(f_wide * g_k) * dt

# Plot
fig, axs = plt.subplots(nrows=2, ncols=2)

axs[0, 0].plot(t_axis, f_narrow)
axs[0, 0].set_title("Narrow in time")
axs[0, 0].set_xlabel("Time [s]")


axs[0, 1].plot(t_axis, f_wide)
axs[0, 1].set_title("Wide in time")
axs[0, 1].set_xlabel("Time [s]")

axs[1, 0].plot(xi_axis, F_narrow.real)
axs[1, 0].set_title("Wide in frequency")
axs[1, 0].set_xlabel("Frequency [Hz]")

axs[1, 1].plot(xi_axis, F_wide.real)
axs[1, 1].set_title("Narrow in frequency")
axs[1, 1].set_xlabel("Frequency [Hz]")

fig.tight_layout()
plt.show()

### C) Discrete time Fourier transform & discrete Fourier transform

The DTFT is defined for $f$ with a discrete domain, through the Poisson summation formula. The DTFT can equivalently be defined as the CTFT on a sampled version of $f$ with sampling interval $\Delta t$. This results in the CTFT of a Dirac comb. Notice that the result is still defined for continuous $\xi$.

$$
\begin{aligned}
F_{1/\Delta t}(\xi) &\triangleq \sum_{n=-\infty }^{\infty } F\left(\xi-n/\Delta t \right) \\
F_{1/\Delta t}(\xi) &\triangleq \mathcal{F} \left\{\sum_{k=-\infty }^{\infty } \Delta t \cdot f(k\Delta t) \cdot \delta (t - k\Delta t) \right\} = \sum_{k=-\infty }^{\infty } \Delta t \cdot f(k\Delta t) \cdot e^{-i2\pi \xi k \Delta t}
\end{aligned}
$$

So when we take the DTFT of a function $f$, we get a sum of shifted versions of the CTFT of $f$, which is a consequence of the Poisson summation. Notice that the index changes from $k$ to $n$ for the two definitions, this is to highlight that the expressions are not equal term per term. So we cannot just calculate the unshifted CTFT by only considering one term of the other expressions. The shift in the frequency domain is $1/\Delta t$. If this shift $1/\Delta t$ is too small, the shifted $F$'s overlap, which is called aliasing. This means that the frequency content cannot be properly assessed and is an important phenomenon as we always work with sampled signals in practice. To eliminate/reduce aliasing, the sampling frequency $1/\Delta t$ should be chosen sufficiently high for the problem at hand. If the non-zero part of $F$ is confined to the interval $\left[ \frac{-1}{2 \Delta t}, \frac{1}{2 \Delta t}\right]$, there is no overlap. This is closely related to the Nyquist-Shannon sampling theorem.

The DTFT assumes that we have an infinite sequence of samples. In practice, when working with numerical computations, we are always restricted to a finite sequence of samples. The truncation of the DTFT to a sum over $n$ samples is referred to as the DFT.

$$
F(\xi) =  \sum_{k=0}^{n-1} \Delta t \cdot f(k\Delta t) \cdot e^{-i2\pi \xi k \Delta t}
$$

This expression is usually restated so that it no longer explicitly refers to an underlying function being sampled, but just to any sequence of data $x_m$. We also change the names of the indices and give the inverse DFT.

$$
\begin{aligned}
X_k &= \sum_{m=0}^{n-1} x_m e^{-i2\pi \frac{mk}{n}} \\
x_m &= \frac{1}{n}\sum_{k=0}^{n-1} X_k e^{i2\pi \frac{mk}{n}}
\end{aligned}
$$

So the DFT transforms one array of length $n$ into another array of the same length. This means that we can represent the DFT as a simple matrix multiplication of computational complexity $\mathcal{O}(n^2)$. If we take $\phi_n = e^{-i2\pi/n}$, we get the following matrix representation:

$$
\begin{bmatrix}
X_1    \\
X_2    \\
X_3    \\
\vdots \\
X_n    \\
\end{bmatrix} = 
\begin{bmatrix}
1      & 1            & 1               & \dots  & 1                \\
1      & \phi_n       & \phi_n^2        & \dots  & \phi_n^{n-1}     \\
1      & \phi_n^2     & \phi_n^4        & \dots  & \phi_n^{2(n-1)}  \\
\vdots & \vdots       & \vdots          & \ddots & \vdots           \\
1      & \phi_n^{n-1} & \phi_n^{2(n-1)} & \dots  & \phi_n^{(n-1)^2} \\
\end{bmatrix}
\begin{bmatrix}
x_1    \\
x_2    \\
x_3    \\
\vdots \\
x_n    \\
\end{bmatrix}
$$

References:
- https://en.wikipedia.org/wiki/Discrete-time_Fourier_transform
- https://en.wikipedia.org/wiki/Nyquist%E2%80%93Shannon_sampling_theorem
- https://en.wikipedia.org/wiki/Discrete_Fourier_transform
- https://en.wikipedia.org/wiki/Poisson_summation_formula
- https://numpy.org/doc/stable/reference/routines.fft.html
- S. L. Brunton and J. N. Kutz, Data-Driven Science and Engineering: Machine Learning, Dynamical Systems, and Control, 2nd ed. Cambridge: Cambridge University Press, 2022. 

In [ ]:
#############################################
# Comparison of DFT and CTFT, with aliasing #
#############################################

# Define function and sampling
npts_t = 8001
T = [-4, 4] # time domain
dt_integration = (T[1]-T[0])/(npts_t-1)
t_axis = np.linspace(T[0], T[1], num=npts_t)

f = np.exp(-t_axis**2) # Fun fact: Gaussians are an eigenfunction of the Fourier transform, so F will also be a Gaussian, with different scaling
def f_func(t):
    return np.exp(-t**2)

dt_sampling = 1 # Try smaller values and see if you can remove the aliasing
npts_sampled = int((T[1]-T[0])/dt_sampling) + 1
t_axis_sampled = np.linspace(T[0], T[1], num=npts_sampled)
f_sampled = np.exp(-t_axis_sampled**2)

# Calculate continuous Fourier transform for comparison
npts_xi = 2001
XI = [-1, 1] # frequency domain
xi_axis = np.linspace(XI[0], XI[1], num=npts_xi)
F = np.zeros(npts_xi, dtype=complex)

for k in np.arange(npts_xi):
    g_k = np.exp(-1j * 2*np.pi * xi_axis[k] * t_axis)
    F[k] = np.sum(f * g_k) * dt_integration

# Calculate DFT of f as a truncated DTFT
K = 4 # We cannot actually take an infinite sum, so we content ourselves with a finite K, notice that for K = 4/dt_sampling and thus k: -4/dt_sampling --> 4/dt_sampling, we end up with the DFT from the definition (we use as many samples)
nk = K*2+1
DFT = np.zeros((nk, npts_xi), dtype=complex)

for i, k in enumerate(np.arange(-K, K+1)):
    g_k = np.exp(-1j * 2*np.pi * xi_axis * k * dt_sampling)
    DFT[i, :] = dt_sampling * f_func(k*dt_sampling) * g_k
DFT = np.sum(DFT, axis=0).flatten()

# Plot
fig, axs = plt.subplots(nrows=2, ncols=1)

axs[0].plot(t_axis, f, label="f", color="blue")
axs[0].scatter(t_axis_sampled, f_sampled, label="f_sampled", color="red")
axs[0].set_xlabel("Time [s]")
axs[0].legend()

axs[1].plot(xi_axis, F.real, label="F", color="blue")
axs[1].plot(xi_axis, DFT.real, label="DFT", color="red")
axs[1].vlines([-1/(2*dt_sampling), 1/(2*dt_sampling)], [-0.1, -0.1], [1.9, 1.9], colors="grey", linestyles="dashed")
axs[1].set_title("DFT vs CTFT")
axs[1].set_xlabel("Frequency [Hz]")
axs[1].legend()

fig.tight_layout()
plt.show()

In [ ]:
###################
# DFT as a matrix #
###################

# Compute the matrix efficiently
n_samples = 256
idxs = np.arange(n_samples)
phi_n = np.exp(-1j*2*np.pi/n_samples)
J, K = np.meshgrid(idxs, idxs)
DFT = np.power(phi_n, J*K)

# Plot the real part
fig, ax = plt.subplots()
fig.suptitle("DFT as a matrix (real part), $F_{256}$")

mat_plot = ax.matshow(DFT.real, cmap="rainbow")
ax.set_xticks([])
ax.set_yticks([])

plt.colorbar(mappable=mat_plot, ax=ax)

fig.tight_layout()
plt.show()

### D) Fast Fourier transform

The FFT is not a separate transform, rather it is an efficient algorithm to calculate the DFT of an array. The main idea behind the speed-up is that for an even number of samples the matrix multiplication can be broken down into 2 matrix multiplications of smaller matrices. If the number of samples is a power of 2, this process can be repeated until you only have to multiply 2 by 2 matrices. If the number of samples is not a power of 2, it can just be padded with zeros. Consider for example the $F_{256}$ matrix plotted in the previous section. We can break down the computations as follows:

$$
X = F_{256}x = 
\begin{bmatrix}
I_{128} & D_{128}  \\
I_{128} & -D_{128} \\
\end{bmatrix}
\begin{bmatrix}
F_{128} & 0       \\
0       & F_{128} \\
\end{bmatrix}
\begin{bmatrix}
x_{even} \\
x_{odd}  \\
\end{bmatrix}
$$

Where $x$ and $X$ are the original data sequence and its DFT respectively, $x_{even}$ and $x_{odd}$ are the samples with even and odd index respectively, $I_{128}$ are identity matrices and $D_{128}$ are defined as:

$$
D_{128} = 
\begin{bmatrix}
1      & 0      & 0        & \dots  & 0            \\
0      & \phi_n & 0        & \dots  & 0            \\
0      & 0      & \phi_n^2 & \dots  & 0            \\
\vdots & \vdots & \vdots   & \ddots & \vdots       \\
0      & 0      & 0        & \dots  & \phi_n^{127} \\
\end{bmatrix}
$$

This reduces the computational complexity from $\mathcal{O}(n^2)$ to $\mathcal{O}\left(n\,log(n)\right)$. When you use software implementations of the FFT, you should pay attention (read the docs!) to normalizations and ordering of the $x$ and $X$ arrays. This algorithm is very versatile, and a few applications are illustrated in the next section.

References:
- S. L. Brunton and J. N. Kutz, Data-Driven Science and Engineering: Machine Learning, Dynamical Systems, and Control, 2nd ed. Cambridge: Cambridge University Press, 2022. 
- https://numpy.org/doc/stable/reference/routines.fft.html

In [ ]:
############################
# Illustration of speed-up #
############################

# Load audio
audio_fragment = "data/audio/good_vibrations_sample.wav"
output = wavfile.read(audio_fragment)
f_sample = output[0]
x = output[1][:, 0] # stereo sound has 2 channels, just select one here
n_samples = x.size
T = (n_samples-1)/f_sample
dt = 1/f_sample
t_axis = np.linspace(0, T, num=n_samples)

print(f"This audio fragment is sampled at {f_sample} Hz!")

# Visualize audio
fig, axs = plt.subplots(nrows=2)

axs[0].plot(t_axis, x)
axs[0].set_xlabel("Time [s]")
axs[0].set_ylabel("Amplitude")
axs[0].grid()
axs[0].set_title("Entire fragment")

plot_start_time = 0
plot_length_time = 0.1
plot_idxs = np.arange(plot_start_time*f_sample, (plot_start_time+plot_length_time)*f_sample+1, dtype=int)

axs[1].plot(t_axis[plot_idxs], x[plot_idxs])
axs[1].set_xlabel("Time [s]")
axs[1].set_ylabel("Amplitude")
axs[1].grid()
axs[1].set_title("Snippet")

fig.tight_layout()
plt.show()

# Time DFT and FFT on the just the 0.1 second audio fragment, as even 1 second already results in huge computation times for the DFT
n_samples_snippet = int(plot_length_time*f_sample)+1

# Calculate DFT
tic = time.perf_counter()
idxs_dft = np.arange(n_samples_snippet)
phi_n = np.exp(-1j*2*np.pi/n_samples_snippet)
J, K = np.meshgrid(idxs_dft, idxs_dft)
DFT = np.power(phi_n, J*K)
X_DFT = DFT @ x[plot_idxs]
toc = time.perf_counter()
execution_time_DFT = toc-tic

# Calculate FFT
tic = time.perf_counter()
X_FFT = np.fft.fft(x[plot_idxs])
toc = time.perf_counter()
execution_time_FFT = toc-tic

f_axis = np.fft.fftfreq(n_samples_snippet) * f_sample

# Print timing results
print(f"\nThe DFT took {execution_time_DFT:.4g} seconds to compute for a sequence of {n_samples_snippet} samples.")
print(f"The FFT took {execution_time_FFT:.4g} seconds to compute for a sequence of {n_samples_snippet} samples.\n")

# Plot results
fig, axs = plt.subplots(nrows=2)

axs[0].set_title("DFT")
axs[0].plot(f_axis, X_DFT.real)
axs[0].set_xlim(-10000, 10000) # Limit to frequencies typically present in music
axs[0].set_xlabel("Frequency [Hz]")
axs[0].grid()

axs[1].set_title("FFT")
axs[1].plot(f_axis, X_FFT.real)
axs[1].set_xlim(-10000, 10000)
axs[1].set_xlabel("Frequency [Hz]")
axs[1].grid()

fig.tight_layout()
plt.show()

### Key messages
- Fourier analysis studies the relationship between the time and frequency domain realizations of functions or data arrays.
- Functions can never be arbitrarily localized in both the time and frequency domain leading to a trade-off in time and frequency resolution.
- Increasing the sampling frequency allows to study a larger frequency domain / reduce aliasing.
- Increasing the duration of a sequence of samples / width of a window leads to a higher frequency resolution.
- In practice, when calculated numerically, any Fourier transform is approximated using the FFT algorithm and appropriate normalization.

## 2) FFT application examples

### A) Amplitude spectrum, power spectrum and phase spectrum

We can use the FFT to calculate the power spectrum of a signal, which we can use to identify prominent frequencies and resonances. Closely related but much less commonly used are the amplitude and phase spectrum. Below we plot the three spectra for a 100 Hz vibration with a bias and a real measurement of the vibrations in a car engine.

In [ ]:
#######################################
# Clean example signal: 100 Hz + bias #
#######################################

# Load data
vibration_filename = "data/vibrations/dc_100_hz.csv"
vibration_data = np.loadtxt(vibration_filename, delimiter=",")
x = vibration_data[:, 1]
T = vibration_data[-1, 0] - vibration_data[0, 0]
n_sample = x.size
dt = T/(n_sample-1)
f_sample = 1/dt
t_axis = np.linspace(0, T, num=n_sample)

# Visualize signal
fig, ax = plt.subplots()

plot_start = 0
plot_npts = 100

ax.plot(t_axis[plot_start:plot_start+plot_npts], x[plot_start:plot_start+plot_npts])
ax.set_title("Vibration signal")
ax.set_ylabel("Amplitude")
ax.set_xlabel("Time [s]")

fig.tight_layout()
plt.show()

print(f"The sampling frequency is {f_sample:.4g} Hz")

# Calculate FFT
X = np.fft.fft(x)
f_axis = np.fft.fftfreq(n_sample) * f_sample

# Only retain positive frequencies
if n_sample//2 != n_sample/2:
    X = X[:n_sample//2+1]
    f_axis = f_axis[:n_sample//2+1]
else:
    X = X[:n_sample//2]
    f_axis = f_axis[:n_sample//2]
df = f_axis[1]

# 1: Power spectrum
power_spec = np.abs(X)**2
power_spec_norm = power_spec / (np.sum(power_spec)*df)

fig, ax = plt.subplots()
ax.plot(f_axis, power_spec_norm)
ax.set_title("Power spectrum")
ax.set_ylabel("Power (normalized)")
ax.set_xlabel("Frequency [Hz]")
fig.tight_layout()
plt.show()

total_norm_power = np.sum(power_spec_norm) * df
print(f"The integral of normalized power is {total_norm_power:.4g}.")

# 2: Amplitude spectrum
ampl_spec = np.abs(X)
ampl_spec_norm = ampl_spec / (np.sum(ampl_spec)*df)

fig, ax = plt.subplots()
ax.plot(f_axis, ampl_spec_norm)
ax.set_title("Amplitude spectrum")
ax.set_ylabel("Amplitude (normalized)")
ax.set_xlabel("Frequency [Hz]")
fig.tight_layout()
plt.show()

# 3: Phase spectrum
phase_spec = np.angle(X)

fig, ax = plt.subplots()
ax.plot(f_axis, phase_spec)
ax.set_title("Phase spectrum")
ax.set_ylabel("Phase [°]")
ax.set_xlabel("Frequency [Hz]")
fig.tight_layout()
plt.show()

In [ ]:
##############################################
# Real example signal: car engine vibrations #
##############################################

# Load data
vibration_filename = "data/vibrations/car_engine.csv"
vibration_data = np.loadtxt(vibration_filename, delimiter=",")
x = vibration_data[:, 1]
T = vibration_data[-1, 0] - vibration_data[0, 0]
n_sample = x.size
dt = T/(n_sample-1)
f_sample = 1/dt
t_axis = np.linspace(0, T, num=n_sample)

# Visualize signal
fig, ax = plt.subplots()

plot_start = 0
plot_npts = 100000

ax.plot(t_axis[plot_start:plot_start+plot_npts], x[plot_start:plot_start+plot_npts])
ax.set_title("Vibration signal")
ax.set_ylabel("Amplitude")
ax.set_xlabel("Time [s]")

fig.tight_layout()
plt.show()

print(f"The sampling frequency is {f_sample:.4g} Hz")

# Calculate FFT
X = np.fft.fft(x)
f_axis = np.fft.fftfreq(n_sample) * f_sample

# Only retain positive frequencies
if n_sample//2 != n_sample/2:
    X = X[:n_sample//2+1]
    f_axis = f_axis[:n_sample//2+1]
else:
    X = X[:n_sample//2]
    f_axis = f_axis[:n_sample//2]
df = f_axis[1]

# Power spectrum
power_spec = np.abs(X)**2
power_spec_norm = power_spec / (np.sum(power_spec)*df)

f_axis_rpm = f_axis * 60
stop_idx = np.argmax(f_axis_rpm > 5000)

fig, ax = plt.subplots()
ax.plot(f_axis_rpm[:stop_idx], power_spec_norm[:stop_idx])
ax.set_title("Power spectrum (close-up of lower frequencies)")
ax.set_ylabel("Power (normalized)")
ax.set_xlabel("Frequency [rpm]")
fig.tight_layout()
plt.show()

In [ ]:
###########################################################
# Real example signal: vibration in response to an impact #
###########################################################

# Load data
vibration_filename = "data/vibrations/UPS_freight_impact.csv"
vibration_data = np.loadtxt(vibration_filename, delimiter=",")
x = vibration_data[:, 1]
T = vibration_data[-1, 0] - vibration_data[0, 0]
n_sample = x.size
dt = T/(n_sample-1)
f_sample = 1/dt
t_axis = np.linspace(0, T, num=n_sample)

# Visualize signal
fig, ax = plt.subplots()

ax.plot(t_axis, x)
ax.set_title("Impact response")
ax.set_ylabel("Amplitude")
ax.set_xlabel("Time [s]")

fig.tight_layout()
plt.show()

print(f"The sampling frequency is {f_sample:.4g} Hz")

# Calculate FFT
X = np.fft.fft(x)
f_axis = np.fft.fftfreq(n_sample) * f_sample

# Only retain positive frequencies
if n_sample//2 != n_sample/2:
    X = X[:n_sample//2+1]
    f_axis = f_axis[:n_sample//2+1]
else:
    X = X[:n_sample//2]
    f_axis = f_axis[:n_sample//2]
df = f_axis[1]

# Power spectrum
power_spec = np.abs(X)**2
power_spec_norm = power_spec / (np.sum(power_spec)*df)

stop_idx = 350

fig, ax = plt.subplots()
ax.plot(f_axis[:stop_idx], power_spec_norm[:stop_idx])
ax.set_title("Power spectrum (close-up of lower frequencies)")
ax.set_ylabel("Power (normalized)")
ax.set_xlabel("Frequency [Hz]")
fig.tight_layout()
plt.show()

### B) Filtering

Most digital filters work by transforming the sequence of samples to the frequency domain by the FFT, then applying some windowing or thresholding function, and transforming it back to the time domain. We illustrate how you may implement a band-pass filter with the FFT.

In [ ]:
####################
# Band-pass filter #
####################

# Define function and add noise
T = 10
f_sample = 1000
n_sample = T*f_sample + 1
dt = 1/f_sample
t_axis = np.linspace(0, T, num=n_sample)

signal = np.sin(2*np.pi*t_axis)*np.exp(-(t_axis-T/2)**2/np.pi)
gaussian_white_noise = 0.1*np.random.randn(*signal.shape)
shot_noise = (np.random.rand(*signal.shape) > 0.99) * np.random.randn(*signal.shape)
x = signal + gaussian_white_noise + shot_noise

# Assume we suspect the relevant frequencies to be in the range [0.5; 5]
X = np.fft.fft(x, norm="ortho")
f_axis = np.fft.fftfreq(n_sample) * f_sample

band = [0.5, 5]
zero_idxs = np.logical_or((np.abs(f_axis) < band[0]), (np.abs(f_axis) > band[1]))
X[zero_idxs] = 0
x_filtered = np.fft.ifft(X, norm="ortho").real

# Plot
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(t_axis, x, label="Noisy signal")
ax.plot(t_axis, signal, label="Underlying signal")
ax.plot(t_axis, x_filtered, label="Filtered signal", linestyle="--")

ax.set_title("Filtering a noisy signal")
ax.set_ylabel("Amplitude")
ax.set_xlabel("Time [s]")
ax.legend()

fig.tight_layout()
plt.show()

### C) Spectral derivatives

Spectral derivatives are derivatives of a sequence of samples, taken by transforming the sequence to the frequency domain, performing the derivation there, and then transforming it back. This is sometimes used to solve PDEs. You should pay attention to edge effects, as illustrated below.

In [ ]:
# Define function
T = 10
f_sample = 1000
n_sample = T*f_sample + 1
dt = 1/f_sample
t_axis = np.linspace(0, T, num=n_sample)

x = np.sin(2*np.pi*t_axis)*np.exp(-(t_axis-T/2)**2/np.pi)
x_not_smooth = np.sin(2*np.pi*t_axis)

### Calculate derivative of noisy signal in different ways
# Mathematically
der_true = 2*np.pi*np.cos(2*np.pi*t_axis)*np.exp(-(t_axis-T/2)**2/np.pi) - np.sin(2*np.pi*t_axis)*np.exp(-(t_axis-T/2)**2/np.pi)*(2/np.pi)*(t_axis-T/2)
der_true_ns = 2*np.pi*np.cos(2*np.pi*t_axis)

# Time domain derivative
der_euler = (x[2:]-x[:-2])/(2*dt)
der_euler_ns = (x_not_smooth[2:]-x_not_smooth[:-2])/(2*dt)

# Spectral derivative
X = np.fft.fft(x, norm="ortho")
X_ns = np.fft.fft(x_not_smooth, norm="ortho")
f_axis = np.fft.fftfreq(n_sample) * f_sample
X_der = X * 2*np.pi*1j*f_axis
X_der_ns = X_ns * 2*np.pi*1j*f_axis
der_fft = np.fft.ifft(X_der, norm="ortho").real
der_fft_ns = np.fft.ifft(X_der_ns, norm="ortho").real

# Plot comparison
fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(10, 5))

axs[0, 0].plot(t_axis, x)
axs[0, 0].set_xlabel("Time [s]")
axs[0, 0].set_title("Smooth signal")

axs[1, 0].plot(t_axis, der_true, label="True")
axs[1, 0].plot(t_axis[1:-1], der_euler, label="Euler", linestyle="--")
axs[1, 0].plot(t_axis, der_fft, label="FFT", linestyle=":")
axs[1, 0].set_xlabel("Time [s]")
axs[1, 0].set_title("Derivatives of smooth signal")
axs[1, 0].legend()

axs[0, 1].plot(t_axis, x_not_smooth)
axs[0, 1].set_xlabel("Time [s]")
axs[0, 1].set_title("Non-smooth signal")

axs[1, 1].plot(t_axis, der_true_ns, label="True")
axs[1, 1].plot(t_axis[1:-1], der_euler_ns, label="Euler", linestyle="--")
axs[1, 1].plot(t_axis, der_fft_ns, label="FFT", linestyle=":")
axs[1, 1].set_xlabel("Time [s]")
axs[1, 1].set_title("Derivatives of non-smooth signal")
axs[1, 1].legend()

fig.tight_layout()
plt.show()

## 3) Time-frequency analysis

### A) The Gabor transform

The Gabor transform (GT) is a special case of a short-time Fourier transform (STFT), so we will first briefly introduce the STFT. The STFT is the CTFT of a signal $x(t)$ multiplied with a windowing function $w(t-\tau)$. Varying $\tau$ places the window over different parts of $x(t)$. This means that the transform provides you with information about the frequency content of a signal around time $\tau$. With this type of transformations, it is possible to perform time-frequency analysis and for example compute spectrograms. The definition is given below, from now on we will use $f$ to denote frequency instead of $\xi$.

$$
\textrm{STFT} \{x(t)\}(\tau, f) \triangleq X(\tau, f) = \int_{-\infty}^{\infty} x(t) w(t-\tau) e^{-i 2 \pi f t} \, dt
$$

Varying the width of the window will affect the resolution in the time and frequency domain. Larger windows lead to higher resolution in the frequency domain, but lower resolution in the time domain, due to the uncertainty principle. The boundary of the uncertainty principle is only reached for Gaussian distributions, which are eigenfunctions of the Fourier transform, meaning that a Gaussian in the time domain will transform into a Gaussian in the frequency domain. This motivates the choice of a Gaussian windowing function, which results in the GT.

$$
\textrm{GT} \{x(t)\}(\tau, f) \triangleq G_{x}(\tau, f) = \sqrt[{4}]{\sigma } \int_{-\infty }^{\infty } x(t) e^{-\sigma \pi (t -\tau)^{2}}e^{-j2 \pi f t} dt
$$

This a mathematically practical transform that can generate usefull insight in the signal. It has one main limitation however, namely that the time-frequency resolution trade-off is fixed for the entire range of the transform. It is possible to adapt this trade-off globally by changing the width of the window ($\sigma$), but you cannot have higher frequency resolution for lower frequencies and higher time resolution for higher frequencies. This is possible with the wavelet transform. The image below illustrates the effect of the window width on the resolution. Left has better time resolution (small $\sigma$), and right has better frequency resolution (large $\sigma$).

<img src="data\images\time_freq_res_gt.svg" style="display: block; margin-left: auto; margin-right: auto"></img>

We will use the `ShortTimeFFT` from SciPy to implement our Gabor transform.

References:
- https://en.wikipedia.org/wiki/Short-time_Fourier_transform
- https://en.wikipedia.org/wiki/Gabor_transform
- https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.ShortTimeFFT.html#scipy.signal.ShortTimeFFT

In [ ]:
##############################################
# Gabor transform with varying window widths #
##############################################

# Define signal to be analyzed
T = [0, 7]
f_sample = 400
n_sample = int(f_sample*(T[1]-T[0])+1)
dt = 1/f_sample
t_axis = np.linspace(T[0], T[1], num=n_sample)

# Example 1: steps in frequency
frequencies = [5, 10, 25, 50, 25, 10, 5]
x = np.zeros_like(t_axis)
for i, freq in enumerate(frequencies):
    idxs = np.arange(i*f_sample, (i+1)*f_sample)
    x[idxs] = np.sin(2*np.pi*freq*t_axis[idxs])

# # Example 2: sine sweep
# f_sweep_max = 50
# ft = f_sweep_max*(t_axis-T[0])/(T[1]-T[0])
# x = np.sin(np.pi*ft*t_axis) # Drop the factor 2, to understand why, see the chirp signal: https://en.wikipedia.org/wiki/Chirp

# Plot signal
fig, ax = plt.subplots()

ax.set_title("Vibration with varying frequency")
ax.plot(t_axis, x)
ax.set_xlabel("Time [s]")

fig.tight_layout()
plt.show()

# Define GTs with varying sigma
T_window = 1
n_window = T_window * f_sample # Window and signal have to be sampled at the same frequency

sigmas = np.array([0.025, 0.05, 0.1, 0.2]) * T_window # Standard deviation of Gaussian window, expressed in time
n_sigmas = sigmas * f_sample # Standard deviation of Gaussian window, expressed in number of samples

over_sampling = 10 # Influences image resolution along frequency axis, not related to the time-frequency resolution as discussed previously! Higher values give higher resolution, should be int in this script.
hop = 1 # Influences image resolution along time axis, not related to the time-frequency resolution as discussed previously! Lower values give higher resolution as this is the time step (in samples) between each application of the windowed FFT.

specs = []
for n_sigma in n_sigmas:
    window = gaussian(n_window, std=n_sigma) # Sampled Gaussian windowing function
    GT = ShortTimeFFT(window, hop, f_sample, mfft=(over_sampling*f_sample)) # Object to compute Gabor transform (STFT) for given window
    spec = np.abs(GT.stft(x))**2 # We take the square of the magnitude of GT(x), this is the spectrogram. We do not plot GT(x) directly as this is a complex matrix. We could plot the real part which would correspond to the time evolution of the amplitude spectrum but this is less common than the spectrogram.
    # spec = GT.spectrogram(x) # Equivalent
    specs.append(spec)


# Getting a visually nice plot out of the transform requires quite a bit of post-processing...


# Prepare to plot: calculate nice axes and aspect
nrows_orig, ncols_orig = specs[0].shape[0], specs[0].shape[1]

max_freq_plot = 100 # maximum frequency plotted, feel free to change this
max_freq_plot = min(max_freq_plot, f_sample//2)
max_freq_idx_plot = max_freq_plot*over_sampling

n_freq_ticks = 6 # number of labels on the frequency axis, feel free to change this
freq_ticks = np.linspace(0, max_freq_idx_plot, num=n_freq_ticks, dtype=int)
freq_tick_labels = np.linspace(0, max_freq_plot, num=n_freq_ticks).astype(str)

n_time_ticks = 8 # number of labels on the time axis, feel free to change this
n_half_window = n_window//2
time_ticks = np.linspace(n_half_window, n_half_window+n_sample, num=n_time_ticks, dtype=int)
time_tick_labels = np.linspace(T[0], T[1], num=n_time_ticks).astype(str)

plot_hw_ratio = 0.4 # height to width ratio of each panel in the plot, feel free to change this
aspect = plot_hw_ratio*ncols_orig/max_freq_idx_plot

# Actually plot
fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(10, 5), layout="constrained")
fig.suptitle(f"Comparison of GT with different $\sigma$ scaling")

for i, spec in enumerate(specs):
    row = i//2
    col = i%2
    axs[row, col].matshow(spec[:max_freq_idx_plot+1], aspect=aspect, origin="lower")

    # Remove standard ticks and add sensible axes
    axs[row, col].tick_params(axis="both", which="both", 
                              top=False, bottom=False, left=False, right=False,
                              labeltop=False, labelbottom=False, labelleft=False, labelright=False)

    axs[row, col].set_xticks(time_ticks)
    axs[row, col].set_xticklabels(time_tick_labels)
    axs[row, col].set_yticks(freq_ticks)
    axs[row, col].set_yticklabels(freq_tick_labels)

    axs[row, col].xaxis.set_ticks_position("bottom")
    axs[row, col].yaxis.set_ticks_position("left")

    # Labels and stuff
    axs[row, col].set_xlabel("Time [s]")
    axs[row, col].set_ylabel("Frequency [Hz]")
    axs[row, col].set_title(f"$\sigma$ = {sigmas[i]} s")

plt.show()

In [ ]:
######################################
# What do good vibrations look like? #
######################################

# Load audio
audio_fragment = "data/audio/good_vibrations_sample.wav"
output = wavfile.read(audio_fragment)
f_sample = output[0]
x = output[1][:, 0] # stereo sound has 2 channels, just select one here

spec, df_spec, dt_spec = st.gabor_spectrogram(x, f_sample, 1, 0.05, freq_bounds=[20, 200], hop=1000)

### B) The wavelet transform

Let's start our discussion of the wavelet transform by stating that the wavelet transform (WT) takes many forms, including continuous and discrete analysis, many different normalizations and many different 'mother wavelets'. There is a general concept, which basically comes down to generalizing the (short-time) Fourier transform, which can be considered a specific case of the wavelet transform. Where Fourier analysis makes use of complex exponentials, wavelet transforms make use of the more general wavelets. These wavelets are more flexible and hence lead to a more customizable trade-off between time and frequency resolution of the transform. We start by briefly considering the continuous wavelet transform (CWT) of a signal $x(t)$, which is given by:

$$
X_{w}(a,b) = \frac{1}{|a|^{1/2}} \int_{-\infty }^{\infty } x(t) \overline{\psi} \left({\frac{t-b}{a}}\right) \, dt
$$

Where $\psi$ is a mother wavelet and $a$ and $b$ are a scaling and shifting parameter, respectively. The mother wavelet and its scaling and shifting parameters can be chosen so that the set of daughter wavelets $\psi \left({\frac{t-b}{a}}\right)$ is non-orthogonal, orthogonal or even orthonormal. There are a number of conditions that a function must satisfy in order to be considered an 'admissible' wavelet. As a rigorous mathematical treatment of the wavelet transform is out of scope for this course, we will just state that if a function looks like a 'pulse', so for example a cosine modulated by a Gaussian, it is generally an admissible wavelet.

The choice of the mother wavelet and the parameters really defines what the transform looks like. If the choice leads to a non-orthogonal set of daughter wavelets, the resulting transform will contain redundant information, which is desirable to get informative, easily interpretable spectrogram plots. If the choice leads to an orthogonal set of daughter wavelets (inner product zero), the resulting transform contains no redundant information, which is desirable for file compression or real-time evaluation.

Notice that where the Gabor transform results in a function of time and frequency, the CWT results in a function of time ($b$) and scale ($a$). Finding the relationship between this scale parameter $a$ and an interpretable frequency is not straightforward and depends on the choice of the mother wavelet. We will qualitatively describe the general relationship between the two quantities, by using the modulated cosine wave $cos(2\pi 10t) e^{-t^2}$ as an example. The Fourier transform of this mother wavelet has a peak at 10 Hertz. Applying an $a$ larger than 1 results in a stretched version of the mother wavelet, with a Fourier transform that has a peak at $10/a$ Hertz, so a lower frequency. This stretched wavelet is also wider (less localized/lower resolution) in the time domain, meaning that the wavelet transform will have a higher frequency resolution for the lower frequencies. If on the other hand $a$ is smaller than 1, the wavelet is compressed and its Fourier transform has a peak at a higher frequency. This compressed wavelet is narrower (more localized/higher resolution) in the time domain, meaning that the wavelet transform will have a lower frequency resolution for the higher frequencies. We can visualize this varying resolution across the time-frequency domain as in the figure below:

<img src="data\images\time_freq_res_wt.svg" style="display: block; margin-left: auto; margin-right: auto"></img>

The choice of wavelet can be tailored to the application. If you are interested in edge detection for example, you can use a boxy wavelet like the Haar wavelet. The (complex) Morlet wavelet transform (MWT) is closely related to the Gabor transform, as it is a complex exponential modulated by a Gaussian window. The only difference between the MWT and the GT, is that for the MWT the window is stretched by $a$, while for the GT the window remains fixed for a given choice of $\sigma$. The Morlet wavelet is a popular choice for wavelet transforms in nonlinear dynamical system analysis. Other popular wavelets include: the Daubechies wavelet (JPEG compression), the Mexican hat wavelet and the derivative of Gaussian (DOG) wavelet to name a few.

These wavelet transforms can be discretized and implemented efficiently in FFT-like algorithms. Some implementations select a logarithmic subsampling of the frequencies to be considered and in doing so achieve a computational complexity of only $\mathcal{O}(n)$. We will implement examples using the PyWavelets library, because it is extensive and well-documented. You may consider ssqueezepy if you are interested in cutting-edge algorithms and computational efficiency.

References:
- I. Daubechies, "The wavelet transform, time-frequency localization and signal analysis," in IEEE Transactions on Information Theory, vol. 36, no. 5, pp. 961-1005, Sept. 1990, doi: 10.1109/18.57199. 
- Torrence, C. and Compo, G. (1998). A Practical Guide to Wavelet Analysis. Bulletin of the American Meteorological Society 79(1) pp. 61-78. Available at: https://journals.ametsoc.org/view/journals/bams/79/1/1520-0477_1998_079_0061_apgtwa_2_0_co_2.xml [Accessed 5 Oct 2026]. 
- https://en.wikipedia.org/wiki/Continuous_wavelet_transform
- https://pywavelets.readthedocs.io/en/latest/ref/cwt.html
- https://github.com/OverLordGoldDragon/ssqueezepy
- https://en.wikipedia.org/wiki/Multiresolution_analysis
- https://en.wikipedia.org/wiki/Morlet_wavelet
- https://en.wikipedia.org/wiki/Wavelet

In [ ]:
#####################################
# Wavelet transform and spectrogram #
#####################################

# Define signal to be analyzed
T = [0, 7]
f_sample = 1000
n_sample = int(f_sample*(T[1]-T[0])+1)
dt = 1/f_sample
t_axis = np.linspace(T[0], T[1], num=n_sample)

# Example 1: steps in frequency
frequencies = [5, 10, 25, 50, 25, 10, 5]
x = np.zeros_like(t_axis)
for i, freq in enumerate(frequencies):
    idxs = np.arange(i*f_sample, (i+1)*f_sample)
    x[idxs] = np.sin(2*np.pi*freq*t_axis[idxs])

# # Example 2: sine sweep
# f_sweep_max = 50
# ft = f_sweep_max*(t_axis-T[0])/(T[1]-T[0])
# x = np.sin(np.pi*ft*t_axis) # Drop the factor 2, to understand why, see the chirp signal: https://en.wikipedia.org/wiki/Chirp

# Plot signal
fig, ax = plt.subplots()

ax.set_title("Vibration with varying frequency")
ax.plot(t_axis, x)
ax.set_xlabel("Time [s]")

fig.tight_layout()
plt.show()

# Choose and plot mother wavelet
wavelet = "cmor2-2" # feel free to change this ["morl", "cmor1.0-1.0", "gaus1", "cgau1", "shan1.0-1.0", "mexh", "fbsp1-1.5-1.0"]
x_wavelet, t_axis_wavelet = pywt.ContinuousWavelet(wavelet).wavefun(10)

fig, ax = plt.subplots()
fig.suptitle(wavelet)

ax.plot(t_axis_wavelet, x_wavelet.real, label="real")
ax.plot(t_axis_wavelet, x_wavelet.imag, label="imag")
ax.set_xlim([-5, 5])
ax.set_ylim([-1, 1.2])
ax.legend()

fig.tight_layout()
plt.show()

### Compute wavelet transform
precision = 12 # needs to be sufficiently high to ensure decent numerical approximation, but choosing a higher value leads to larger computation times

# Define desired frequency axis of the transformed signal
freq_scale = "log10" # "linear", "log10", "log2", feel free to change this
freq_bounds = [1, 100] # lower bound must be larger than 0 (hard constraint), upperbound should probably stay below f_sample for interpretability
n_freqs = 512 # number of frequencies to calculate the CWT for

if freq_scale == "linear":
    freqs_of_interest = np.linspace(freq_bounds[0]/f_sample, freq_bounds[1]/f_sample, num=n_freqs) # linear, frequencies to evaluate CWT for
elif freq_scale == "log10":
    start = np.log10(freq_bounds[0]/f_sample)
    end = np.log10(freq_bounds[1]/f_sample)
    freqs_of_interest = np.logspace(start, end, num=n_freqs) # logarithmic, base 10
elif freq_scale == "log2":
    start = np.log2(freq_bounds[0]/f_sample)
    end = np.log2(freq_bounds[1]/f_sample)
    freqs_of_interest = np.logspace(start, end, num=n_freqs, base=2) # logarithmic, base 2

# Actually calculate CWT
scales = pywt.frequency2scale(wavelet, freqs_of_interest, precision=precision) # transform frequencies to corresponding scales for a given wavelet
coefs, freq_axis = pywt.cwt(x, scales, wavelet, sampling_period=dt, method="fft", precision=precision) # evaluate CWT for given wavelet and scales
spec = np.abs(np.sqrt(dt)*coefs)**2 # Compute spectrogram

# Plot the spectrogram based on the transform
plot_hw_ratio = 0.4
aspect = plot_hw_ratio*spec.shape[1]/n_freqs

fig, ax = plt.subplots(figsize=(10, 5))

ax.set_title("WT-based spectrogram")
mappable = ax.pcolormesh(t_axis, freq_axis, spec)
# mappable = ax.pcolormesh(t_axis, freq_axis, np.log(spec)) # try this for a funky plot

if freq_scale == "log10":
    ax.set_yscale("log")
elif freq_scale == "log2":
    ax.set_yscale("log", base=2)

clrbr = plt.colorbar(mappable=mappable, ax=ax)

ax.set_box_aspect(plot_hw_ratio)

ax.set_xlabel("Time [s]")
ax.set_ylabel("Frequency [Hz]")

fig.tight_layout()
plt.show()

In [ ]:
######################################
# What do good vibrations look like? #
######################################

# Load audio
audio_fragment = "data/audio/good_vibrations_sample.wav"
output = wavfile.read(audio_fragment)
f_sample = output[0]
x = output[1][:, 0] # stereo sound has 2 channels, just select one here

# Calculate spectrum
# You can try varying the parameters of the Morlet wavelet to obtain a nice plot, "cmor1-1" is pretty shitty, 
# but "cmor4-4" already gives a nice result for example. 
# You always have to tune the mother wavelet to the application. 
# You can check the PyWavelets documentation to see what the parameters mean, the first number X in "cmorX-Y", 
# is very similar to the sigma parameter in the Gabor transform. 
spec, freq_axis = st.wavelet_spectrogram(x, f_sample, "cmor8-5", freq_bounds=[20, 1000], n_freqs=1024, plot_scale=None)

In [ ]:
##########################################
# What do devilish vibrations look like? #
##########################################

# Load audio
audio_fragment = "data/audio/formula_sample.wav" # The producer hid something in this song that we sampled
output = wavfile.read(audio_fragment)
f_sample = output[0]
x = output[1][:, 0]

# Here the Gabor transform is preferred over the wavelet transform because the wavelet transform lacks the 'hop' argument,
# leading to very long computation times for signals with high fs and of long duration.
spec, df_spec, dt_spec = st.gabor_spectrogram(x, f_sample, 1, 0.0125, freq_bounds=[60, 20000], hop=1000, freq_scale="log10", plot_scale="log10")

### Key messages
- The Gabor and wavelet transform are used to visualize/study how the frequency content of a signal changes over time.
- The Gabor transform is basically a windowed Fourier transform of signal, where the window moves over the signal.
- The wavelet transform is a generalization of the (short time) Fourier transform allowing for more tuning/flexibility.
- Different transforms make different trade-offs in time-frequency resolution.
- The window or mother wavelet has to be tuned to the application.

## 4) Gabor and wavelet transform application examples

In [ ]:
#######################################
# Clean example signal: 100 Hz + bias #
#######################################
reload(st)
# Load data
vibration_filename = "data/vibrations/dc_100_hz.csv"
vibration_data = np.loadtxt(vibration_filename, delimiter=",")
x = vibration_data[:, 1]
T = vibration_data[-1, 0] - vibration_data[0, 0]
n_sample = x.size
dt = T/(n_sample-1)
f_sample = 1/dt
t_axis = np.linspace(0, T, num=n_sample)

# Visualize signal
fig, ax = plt.subplots()

plot_start = 0
plot_npts = 100

ax.plot(t_axis[plot_start:plot_start+plot_npts], x[plot_start:plot_start+plot_npts])
ax.set_title("Vibration signal")
ax.set_ylabel("Amplitude")
ax.set_xlabel("Time [s]")

fig.tight_layout()
plt.show()

# Calculate wavelet spectrogram
_, _ = st.wavelet_spectrogram(x, f_sample, "cmor4-4", downsample=False, 
                              n_freqs=512, freq_bounds=[10, 200], freq_scale="log10")

In [ ]:
##############################################
# Real example signal: car engine vibrations #
##############################################
reload(st)
# Load data
vibration_filename = "data/vibrations/car_engine.csv"
vibration_data = np.loadtxt(vibration_filename, delimiter=",")
x = vibration_data[:, 1]
T = vibration_data[-1, 0] - vibration_data[0, 0]
n_sample = x.size
dt = T/(n_sample-1)
f_sample = 1/dt
t_axis = np.linspace(0, T, num=n_sample)

# Visualize signal
fig, ax = plt.subplots()
plot_start = 0
plot_npts = 100000
ax.plot(t_axis[plot_start:plot_start+plot_npts], x[plot_start:plot_start+plot_npts])
ax.set_title("Vibration signal")
ax.set_ylabel("Amplitude")
ax.set_xlabel("Time [s]")
fig.tight_layout()
plt.show()

# Calculate Gabor spectrogram
_, _, _ = st.gabor_spectrogram(x, f_sample, 1, 0.05, 
                               hop=1000, over_sampling=1, 
                               freq_bounds=[10, 1000], freq_scale="log10", plot_scale="log10")

In [ ]:
###########################################################
# Real example signal: vibration in response to an impact #
###########################################################

# Load data
vibration_filename = "data/vibrations/UPS_freight_impact.csv"
vibration_data = np.loadtxt(vibration_filename, delimiter=",")
x = vibration_data[:, 1]
T = vibration_data[-1, 0] - vibration_data[0, 0]
n_sample = x.size
dt = T/(n_sample-1)
f_sample = 1/dt
t_axis = np.linspace(0, T, num=n_sample)

# Visualize signal
fig, ax = plt.subplots()

ax.plot(t_axis, x)
ax.set_title("Impact response")
ax.set_ylabel("Amplitude")
ax.set_xlabel("Time [s]")

fig.tight_layout()
plt.show()

# Calculate wavelet spectrogram
_, _ = st.wavelet_spectrogram(x, f_sample, "cmor0.5-1", downsample=False, 
                              n_freqs=512, freq_bounds=[30, 200], freq_scale="log10")
_, _ = st.wavelet_spectrogram(x, f_sample, "cmor0.5-4", downsample=False, 
                              n_freqs=512, freq_bounds=[30, 200], freq_scale="log10")

In [ ]:
################################################################
# Real example signal: aircraft wing vibration during take-off #
################################################################
reload(st)
# Load data
vibration_filename = "data/vibrations/aircraft_takeoff.csv"
vibration_data = np.loadtxt(vibration_filename, delimiter=",")
x = vibration_data[:, 1]
T = vibration_data[-1, 0] - vibration_data[0, 0]
n_sample = x.size
dt = T/(n_sample-1)
f_sample = 1/dt
t_axis = np.linspace(0, T, num=n_sample)

# Visualize signal
fig, ax = plt.subplots()

ax.plot(t_axis, x)
ax.set_title("Wing vibrations")
ax.set_ylabel("Amplitude")
ax.set_xlabel("Time [s]")

fig.tight_layout()
plt.show()

# Calculate Gabor spectrogram
_, _, _ = st.gabor_spectrogram(x, f_sample, 20, 2, 
                               hop=1000, over_sampling=1, 
                               freq_bounds=[10, 1000], freq_scale="log10", plot_scale="log10")